# 02. Fine-tune model T for every Congress, 104th–119th

Uses the settings chosen by the sweep (read from the sweep's parent run in MLflow, `optuna/best.json`) without change, with seed 20261105 for every Congress, plus seeds 1 and 2 for the 115th and 119th (`docs/preregistration.md` §5). Each run logs the test accuracy with its cluster-bootstrap interval, the accuracy with party names masked (S1), and the per-turn test predictions as an artifact. The loop is resumable: a Congress and seed already logged as finished is skipped.

At the end, the 119th-Congress model (seed 20261105) is uploaded to the private Hugging Face repository `kvenanzi/congress-party-classifier`.

**Before running:** L4 GPU; the same four secrets as notebook 01; notebook 01 finished. Expect about 15 minutes per run, 20 runs, about 5 hours and 25 compute units at the L4 rate. It can be run across several sessions.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L

## 2. Get the code
The repository is private until the post is published, so the clone uses the `GITHUB_TOKEN` secret if it is set.

In [ ]:
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
url = f"https://{'kvenanzi:' + token + '@' if token else ''}github.com/kvenanzi/congress-speech.git"   # fine-grained PAT, Contents: read
if not os.path.isdir("congress-speech"):
    subprocess.run(["git", "clone", "-q", url], check=True)
else:
    subprocess.run(["git", "-C", "congress-speech", "pull", "-q"], check=True)
sys.path.insert(0, "congress-speech")
!git -C congress-speech log --oneline -1

## 3. Install what Colab lacks
MLflow is pinned to the server's version.

In [ ]:
%pip install -q "mlflow==3.14.0" optuna

## 4. Connect to MLflow and Hugging Face
Read the compute-unit rate for this runtime from Colab's resources panel and type it below; it is stored on every run as `cu_estimate`.

In [ ]:
CU_PER_HOUR = None   # e.g. 4.8 for an L4: read it from the resources panel
assert CU_PER_HOUR, "set CU_PER_HOUR from Colab's resources panel"

os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.withinnoise.dev"
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("MLFLOW_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("MLFLOW_PASSWORD")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import torch, transformers
from congress_speech import tracking, finetune, stats
mlflow = tracking.setup()
COMMIT = subprocess.run(["git", "-C", "congress-speech", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
RUNTIME = "colab-" + torch.cuda.get_device_name(0).split()[-1]
print(RUNTIME, "torch", torch.__version__, "transformers", transformers.__version__, "commit", COMMIT)

## 5. Data, settings, and a dry run

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

def sample(congress):
    path = hf_hub_download("kvenanzi/congressional-record-speeches-1995-2026", f"samples/govinfo_{congress}.parquet", repo_type="dataset")
    return pd.read_parquet(path)

In [ ]:
import json
sweep = mlflow.search_runs(experiment_names=[tracking.EXPERIMENT], output_format="list",
                           filter_string="tags.stage = 'sweep-parent' and tags.`optuna.study` = 'congress-speech-sweep-01'")
assert len(sweep) == 1, "run notebook 01 first"
best = json.loads(open(mlflow.artifacts.download_artifacts(run_id=sweep[0].info.run_id, artifact_path="optuna/best.json")).read())
BEST = best["best_params"]
print(BEST)

In [ ]:
# A tiny run first, so that a mistake costs a minute rather than compute units. Not logged.
s = sample(108)
tiny = pd.concat([g.head(n) for (sp, g), n in zip(s.groupby("split"), [32, 64, 32])])  # test, train, val
_, _, r, pred = finetune.train(tiny, finetune.Params(epochs=1, max_len=128, batch_size=16))
print(r["val_acc"], len(pred), pred.columns.tolist())

## 6. The runs

In [ ]:
import tempfile, time
from pathlib import Path

KEEP = (119, 20261105)          # the model published on Hugging Face; trained last, so the upload below
                                # runs in the same session that trained it
JOBS = [(c, 20261105) for c in range(104, 119)] + [(115, 1), (115, 2), (119, 1), (119, 2), KEEP]

for congress, seed in JOBS:
    if tracking.finished_runs(congress=str(congress), model="modernbert", stage="final", seed=str(seed)):
        print(congress, seed, "already logged"); continue
    p = finetune.Params(**BEST, seed=seed)
    data = sample(congress)
    t0 = time.time()
    with mlflow.start_run(run_name=f"congress-{congress}-modernbert-s{seed}") as run:
        mlflow.set_tags({"congress": congress, "model": "modernbert", "stage": "final", "seed": seed,
                         "source": "govinfo", "runtime": RUNTIME, "git_commit": COMMIT})
        mlflow.log_params(vars(p))
        model, tok, r, pred = finetune.train(data, p, log=lambda m, step: mlflow.log_metrics(m, step=step))
        ci = stats.accuracy_ci(pred.correct.to_numpy(), pred.bioguide)
        cim = stats.accuracy_ci(pred.correct_masked.to_numpy(), pred.bioguide)
        hours = (time.time() - t0) / 3600
        mlflow.log_metrics({"val_acc": r["val_acc"], "test_acc": ci["acc"], "test_acc_lo": ci["lo"],
                            "test_acc_hi": ci["hi"], "test_acc_masked": cim["acc"], "test_acc_masked_lo": cim["lo"],
                            "test_acc_masked_hi": cim["hi"], "hours": hours})
        mlflow.set_tag("cu_estimate", round(hours * CU_PER_HOUR, 2))
        with tempfile.TemporaryDirectory() as d:
            pred.to_parquet(Path(d) / "test_predictions.parquet", index=False)
            mlflow.log_artifacts(d)
        if (congress, seed) == KEEP:
            model.save_pretrained("/content/model_119", safe_serialization=True)
            tok.save_pretrained("/content/model_119")
            (Path("/content/model_119") / "mlflow_run_id.txt").write_text(run.info.run_id)
    print(f"{congress} s{seed}: test {ci['acc']:.3f} ({ci['lo']:.3f}-{ci['hi']:.3f}), masked {cim['acc']:.3f}, {hours*60:.0f} min")
    del model; torch.cuda.empty_cache()

## 7. Upload the 119th-Congress model (private)
The model card is written later from the finished results by `scripts/06_publish_hf.py`; this cell uploads the weights and tokenizer only.

In [ ]:
from huggingface_hub import HfApi
assert Path("/content/model_119").exists(), "the 119th-Congress model was trained in an earlier session; rerun it by deleting its MLflow run, or skip the upload"
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo("kvenanzi/congress-party-classifier", private=True, exist_ok=True)
api.upload_folder(folder_path="/content/model_119", repo_id="kvenanzi/congress-party-classifier",
                  commit_message=f"ModernBERT-base fine-tuned on the 119th Congress, code {COMMIT}")
print("https://huggingface.co/kvenanzi/congress-party-classifier")